In [ ]:
# ---- 1. mount + locate the file ----
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
# os.environ["GEMINI_API_KEY"]   = userdata.get("GEMINI_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
# print(build_clients().keys())

In [ ]:
!pip install openai pandas pyarrow -q

In [ ]:
#!/usr/bin/env python3
"""
tick_screen_prompt_v3.py  —  Google Colab
=========================================
Drop-in replacement for the ORIGINAL screening prompt. Same output columns as
before, so nothing downstream breaks:

    gpt_label  gpt_conf  gpt_reason  deepseek_label  deepseek_conf
    deepseek_reason  llm_label

plus three new columns that carry the extra information for free:

    gpt_type  deepseek_type  llm_type  llm_agree

WHAT CHANGED (v2 -> v3)
  OLD question: "is this tick-relevant?"  -> accepted Lyme trials with no tick.
  NEW question: "are TICKS the main object of study?"

  DECISION RULE (binary):
    TICK_FOCUSED      = ticks are what the study is ABOUT. Either physical tick
                        material was used (whole ticks, body parts, extracts,
                        cell lines), OR ticks were counted/measured, OR pathogens
                        were detected IN ticks, OR ticks are the modelled subject
                        (distribution/suitability/risk), OR the paper is a review
                        whose main subject is ticks.
    NOT_TICK_FOCUSED  = ticks are not the object: disease-in-patients studies and
                        clinical reviews of a tick-borne disease, ticks mentioned
                        only as background, or not about ticks at all.

  v3.1 CHANGE: `model_only` -> `tick_model` and moved to the FOCUSED side; new
  `tick_review` type for reviews whose main subject is ticks. The criterion is
  therefore "ticks as object of study", broader than "physical tick material".
  Old cached records with type "model_only" are remapped automatically.

DESIGN: the model names the TYPE; the binary label is DERIVED from the type in
code. Models are more consistent describing what a paper did than judging
yes/no, and this removes label/reason contradictions.

SETUP
  !pip install openai pandas pyarrow -q     # Colab secrets: OPENAI_API_KEY, DEEPSEEK_API_KEY
"""

import json, os, re, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd

# --------------------------------------------------------------------------- #
# read secrets ONCE, up front (avoids the mid-run Colab TimeoutException)
# --------------------------------------------------------------------------- #
try:
    from google.colab import userdata
    for _k in ("OPENAI_API_KEY", "DEEPSEEK_API_KEY"):
        try:
            _v = userdata.get(_k)
            if _v: os.environ[_k] = _v
        except Exception:
            pass
except Exception:
    pass

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/screened_llm.parquet"
OUT_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass2/screened_llm_v3.parquet"
OUT_CSV     = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass2/screened_llm_v3.csv"
LOCAL_CACHE = "/content/screen_v3_cache.jsonl"
DRIVE_CACHE = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass2/screen_v3_cache.jsonl"
BATCH        = 8          # smaller batch = fewer truncated JSON responses
WORKERS      = 5
MAX_TOKENS   = 6000       # generous: reasons are short but 8 records add up
BACKUP_EVERY = 10


# =========================================================================== #
# ============================  THE PROMPT  ================================= #
# =========================================================================== #
SYSTEM_PROMPT = """You screen scientific abstracts for a study of TICK research.

THE ONE QUESTION: did this study actually USE, EXAMINE, or MEASURE physical ticks or tick-derived material?

It is NOT enough that the paper is about a tick-borne disease. A clinical trial for Lyme disease in patients, where no tick was collected or examined, is NOT tick-focused.

For each abstract, choose exactly ONE type from this list.

=== TYPES THAT COUNT AS TICK-FOCUSED (physical tick material was used) ===
tick_organism        Whole ticks are studied: taxonomy, morphology, identification, collection/field sampling, biology, life cycle, behaviour, physiology, genomics/transcriptomics of ticks, tick microbiome, laboratory tick colonies, tick feeding or xenodiagnosis experiments.
tick_body_part       A tick body part or tissue is studied: salivary glands, midgut, ovary, cuticle, haemolymph, synganglion.
tick_derived         Material or molecules derived from ticks: saliva, salivary extracts, tick proteins/peptides, tick cell lines (IDE8, ISE6, BME26), tick antigens and anti-tick vaccines (e.g. Bm86), tick-derived anticoagulants or immunomodulators.
tick_count_prevalence  Ticks are counted or measured: abundance, questing density, infestation prevalence, tick burden on hosts, seasonal dynamics, distribution based on ACTUAL collected specimens.
pathogen_in_tick     A pathogen or microorganism is detected, isolated, sequenced, or quantified IN or FROM ticks (the tick is the sample): PCR screening of ticks, infection rates in ticks, pathogen isolation from ticks.
tick_control         An intervention is applied to or tested against real ticks: acaricides, acaricide resistance, repellents tested on ticks, biological control, treated hosts where tick burden is measured.
tick_model           Ticks are the modelled subject, even though no ticks were handled: species distribution or niche models, climate-suitability projections, tick risk maps, population or transmission simulations where the TICK is what is modelled.
tick_review          A review, meta-analysis, or synthesis whose MAIN SUBJECT is ticks: tick biology, tick taxonomy, tick distribution, tick control, tick-borne pathogen ecology in ticks. The paper must be ABOUT ticks, not about a human disease.

=== TYPES THAT DO NOT COUNT ===
disease_only         Tick-borne disease in humans or animals with NO tick examined and ticks not the subject: clinical studies, patient serosurveys, case reports, outbreak investigations, human/animal diagnostics, treatment or pathogen-vaccine trials in hosts, and clinical reviews of a tick-borne disease.
mention_only         Ticks appear only as background or in passing (e.g. "transmitted by ticks") and are not studied.
not_tick             Not about ticks at all: mites or other arthropods, or "tick" used in another sense (checkmark, time unit).

=== RULES ===
- Detected a pathogen in ticks -> pathogen_in_tick (even if hosts were also sampled).
- Ticks counted on animals -> tick_count_prevalence. Only host blood/serology tested, no ticks -> disease_only.
- Tick cell line or tick tissue in vitro -> tick_derived or tick_body_part, NOT tick_organism.
- Vaccine made FROM a tick molecule -> tick_derived. Vaccine against the PATHOGEN given to hosts -> disease_only.
- A review or meta-analysis: judge the OBJECT reviewed. A review whose main subject is ticks -> tick_review; a clinical review of a human tick-borne disease -> disease_only.
- Distribution work: based on ticks actually collected -> tick_count_prevalence; modelled or projected without handling ticks -> tick_model.
- A model of DISEASE incidence in humans (not of the tick) -> disease_only. A model of TICK range, abundance, or suitability -> tick_model.
- If the abstract is missing, too short, or genuinely ambiguous, use type "uncertain".

=== OUTPUT ===
Return ONLY a JSON array, one object per input, in the SAME order:
[{"type":"...","conf":0.0-1.0,"reason":"..."}]
- "type": exactly one value from the lists above, or "uncertain".
- "conf": your calibrated confidence, 0.0-1.0. Use low values honestly; do not inflate.
- "reason": ONE short sentence (max 20 words) quoting or paraphrasing what in the abstract decided it.
No prose, no markdown fences, no extra keys."""

# --------------------------------------------------------------------------- #
# type -> binary label is derived in CODE (not left to the model)
# --------------------------------------------------------------------------- #
FOCUSED_TYPES = {"tick_organism", "tick_body_part", "tick_derived",
                 "tick_count_prevalence", "pathogen_in_tick", "tick_control",
                 "tick_model", "tick_review"}
NOT_FOCUSED_TYPES = {"disease_only", "mention_only", "not_tick"}
# kept so old cached records using the previous name still resolve correctly
LEGACY_TYPE_MAP = {"model_only": "tick_model"}
ALL_TYPES = FOCUSED_TYPES | NOT_FOCUSED_TYPES | {"uncertain"}

def canon_type(t):
    t = (t or "").strip().lower()
    return LEGACY_TYPE_MAP.get(t, t)

def type_to_label(t):
    t = canon_type(t)
    if t in FOCUSED_TYPES: return "TICK_FOCUSED"
    if t in NOT_FOCUSED_TYPES: return "NOT_TICK_FOCUSED"
    return "UNCERTAIN"


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1]


# =========================================================================== #
# LLM calls                                                                    #
# =========================================================================== #
def _client(which):
    from openai import OpenAI
    if which == "deepseek":
        return OpenAI(api_key=os.environ.get("DEEPSEEK_API_KEY"),
                      base_url="https://api.deepseek.com"), "deepseek-v4-flash"
    return OpenAI(api_key=os.environ.get("OPENAI_API_KEY")), "gpt-4o-mini"

def _call(client, model, batch):
    lines = [f'{i}. TITLE: {S(r.get("oa_title"))[:500]}\nABSTRACT: {S(r.get("oa_abstract"))[:3500]}'
             for i, r in enumerate(batch, 1)]
    user = "Screen each record:\n\n" + "\n\n".join(lines)
    for attempt in range(4):
        try:
            resp = client.chat.completions.create(
                model=model, temperature=0, max_tokens=MAX_TOKENS,
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": user}])
            txt = resp.choices[0].message.content.strip()
            txt = re.sub(r"^```(?:json)?|```$", "", txt, flags=re.MULTILINE).strip()
            arr = json.loads(txt)
            if isinstance(arr, list) and len(arr) == len(batch):
                return arr
            # length mismatch -> retry once more before giving up
        except json.JSONDecodeError:
            time.sleep(1 + attempt)
        except Exception:
            time.sleep(2 ** attempt)
    return [None] * len(batch)

def norm(o):
    if not isinstance(o, dict): return None
    t = canon_type(S(o.get("type")))
    if t not in ALL_TYPES: t = "uncertain"
    try:
        conf = float(o.get("conf"))
    except (TypeError, ValueError):
        conf = 0.0
    return {"type": t, "conf": round(max(0.0, min(1.0, conf)), 3),
            "reason": S(o.get("reason"))[:300]}


# =========================================================================== #
# cache                                                                        #
# =========================================================================== #
def restore_cache():
    import shutil
    if Path(DRIVE_CACHE).exists() and not Path(LOCAL_CACHE).exists():
        Path(LOCAL_CACHE).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    done = {}
    if Path(LOCAL_CACHE).exists():
        for line in open(LOCAL_CACHE):
            try:
                d = json.loads(line); done[d["oa_id"]] = d
            except Exception: pass
    print(f"[cache] {len(done):,} already screened", file=sys.stderr)
    return done

def backup_cache():
    import shutil
    try:
        Path(DRIVE_CACHE).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(LOCAL_CACHE, DRIVE_CACHE)
    except Exception as e:
        print(f"[cache] backup failed: {e}", file=sys.stderr)


# =========================================================================== #
# main                                                                         #
# =========================================================================== #
def run(df, cache):
    oa_c, oa_m = _client("openai"); ds_c, ds_m = _client("deepseek")
    todo = [r for _, r in df.iterrows() if short_id(r.get("oa_id")) not in cache]
    print(f"[screen] {len(todo):,} records to screen", file=sys.stderr)
    if not todo: return cache
    batches = [todo[i:i + BATCH] for i in range(0, len(todo), BATCH)]
    fh = open(LOCAL_CACHE, "a")

    def work(batch):
        a = _call(oa_c, oa_m, batch); b = _call(ds_c, ds_m, batch)
        rows = []
        for rec, ra, rb in zip(batch, a, b):
            na, nb = norm(ra), norm(rb)
            g = na or {"type": "", "conf": 0.0, "reason": ""}
            d = nb or {"type": "", "conf": 0.0, "reason": ""}
            gl, dl = type_to_label(g["type"]), type_to_label(d["type"])
            agree = bool(g["type"] and d["type"] and gl == dl)
            # consensus label; if the two binary labels differ -> UNCERTAIN
            if gl == dl and gl != "UNCERTAIN":
                llm_label, llm_type = gl, (g["type"] if g["type"] == d["type"] else g["type"])
            elif gl != "UNCERTAIN" and dl == "UNCERTAIN":
                llm_label, llm_type = gl, g["type"]
            elif dl != "UNCERTAIN" and gl == "UNCERTAIN":
                llm_label, llm_type = dl, d["type"]
            else:
                llm_label, llm_type = "UNCERTAIN", "uncertain"
            rows.append({"oa_id": short_id(rec.get("oa_id")),
                         "gpt_label": gl, "gpt_conf": g["conf"], "gpt_reason": g["reason"],
                         "gpt_type": g["type"],
                         "deepseek_label": dl, "deepseek_conf": d["conf"],
                         "deepseek_reason": d["reason"], "deepseek_type": d["type"],
                         "llm_label": llm_label, "llm_type": llm_type,
                         "llm_agree": agree,
                         "type_agree": bool(g["type"] and g["type"] == d["type"])})
        return rows

    done = 0
    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futs = [ex.submit(work, b) for b in batches]
            for fut in as_completed(futs):
                for d in fut.result():
                    fh.write(json.dumps(d, ensure_ascii=False) + "\n")
                    cache[d["oa_id"]] = d
                fh.flush(); os.fsync(fh.fileno())
                done += 1
                if done % BACKUP_EVERY == 0:
                    backup_cache()
                    print(f"[screen] {done:,}/{len(batches):,} batches", file=sys.stderr)
    finally:
        fh.close(); backup_cache()
    return cache


if __name__ == "__main__":
    #df = pd.read_parquet(IN_PARQUET)

    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]

    #df = df.iloc[1:100]
    #df = df.sample(100)

    print(f"[in] {len(df):,} rows", file=sys.stderr)
    cache = restore_cache()
    cache = run(df, cache)

    res = pd.DataFrame(cache.values())
    df["_sid"] = df["oa_id"].map(short_id)
    out = df.merge(res, left_on="_sid", right_on="oa_id", how="left",
                   suffixes=("", "_v3")).drop(columns=["_sid"])
    out.to_parquet(OUT_PARQUET, index=False)
    out.to_csv(OUT_CSV, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)

    print("\n===== llm_label =====")
    print(out["llm_label"].value_counts().to_string())
    print("\n===== llm_type =====")
    print(out["llm_type"].value_counts().to_string())
    print(f"\n[reliability] binary label agreement (GPT vs DeepSeek): "
          f"{out['llm_agree'].mean():.1%}")
    print(f"[reliability] exact TYPE agreement: {out['type_agree'].mean():.1%}")
    both = out[out["gpt_type"].ne("") & out["deepseek_type"].ne("")]
    print(f"[coverage] both models answered: {len(both):,}/{len(out):,} "
          f"({len(both)/max(len(out),1):.1%})  <-- watch this; low = JSON failures")

In [ ]:
out.head(60)

In [ ]:
out.sample(60)

In [ ]:

out.to_csv(OUT_CSV, index=False)

In [ ]:
import re

# Function to clean illegal XML characters
def clean_excel_text(text):
    if isinstance(text, str):
        # Remove control characters and other illegal XML characters
        # U+0000 to U+0008, U+000B, U+000C, U+000E to U+001F, U+FFFE, U+FFFF
        # and potentially others depending on strictness.
        # A common approach is to keep only valid XML 1.0 characters.
        # This regex keeps valid characters (printable ASCII and common non-ASCII safe chars).
        # The \x1f character is explicitly mentioned in the traceback
        # Updated to use a stricter regex including C1 control characters (\x7f-\x9f)
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text) # Remove common illegal control characters and C1 controls
        # Also consider removing non-breaking spaces or other problematic unicode if they cause issues
        text = text.replace('\xa0', ' ') # Replace non-breaking space with regular space
    return text

# Apply the cleaning function to all string columns in the DataFrame
for col in out.select_dtypes(include=['object']).columns:
    out[col] = out[col].apply(clean_excel_text)

out.to_excel("/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/tickclass2/screened_llm_v3.xlsx", index=False)